# 10 — Independently verify attacking orientation and calculate signed geometry

**Scientific objective:** Compare all 80 goalkeeper-based provisional directions with SkillCorner's native `match.json → home_team_side`, which records home-team attacking direction for first and second halves. Build direction-normalized *geometric* teammate targets **only if all assignments agree**.

A signed change in X describes endpoint progression. It does **not** establish that the pass is feasible, was attempted, successful, or would have progressed play.

**Requirements:** `outputs/P2_attacking_direction_gate.json` and the 20 `outputs/P1_canonical_decisions_match_{mid}_ROLES.jsonl` files, previously produced by notebooks 09 and 05. Does not read continuous tracking, recompute event labels, or use observed outcomes as predictors.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, math

ROOT=next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Open notebook from within SkillCorner repository.')
OUT=ROOT/'outputs'
diag=json.loads((OUT/'P2_attacking_direction_gate.json').read_text())
assert diag['n_matches']==20 and len(diag['all_rows'])==80
print('Loaded 80 diagnostic team-period rows')


Loaded 80 diagnostic team-period rows


## 1 — Reference metadata versus goalkeeper hypothesis

The canonical interpretation is: `left_to_right` means attacking toward increasing tracking X; `right_to_left` means decreasing X. Each away team's direction is the opposite of the home team's direction in the corresponding half. The source metadata is the primary direction label; goalkeeper medians are an independent quality check.


In [2]:
def sign_direction(label):
    return {'left_to_right':1,'right_to_left':-1}.get(label)

results=[]
metadata_map={}
for mid in sorted({str(x['match_id']) for x in diag['all_rows']}):
    meta=json.loads((ROOT/'data'/'matches'/mid/f'{mid}_match.json').read_text())
    sides=meta.get('home_team_side')
    if not isinstance(sides,list) or len(sides)!=2:
        raise AssertionError(f'{mid}: missing/unexpected home_team_side: {sides!r}')
    home=str(meta['home_team']['id'])
    away=str(meta['away_team']['id'])
    for period in (1,2):
        hs=sign_direction(sides[period-1])
        if hs is None: raise AssertionError(f'{mid}: unknown orientation {sides[period-1]!r}')
        metadata_map[(mid,period,home)]=hs
        metadata_map[(mid,period,away)]=-hs
for entry in diag['all_rows']:
    key=(str(entry['match_id']),int(entry['period']),str(entry['team_id']))
    ref=metadata_map.get(key)
    proposed=entry.get('proposed_attack_x_sign')
    results.append({'match_id':key[0],'period':key[1],'team_id':key[2],
      'metadata_sign':ref,'goalkeeper_provisional_sign':proposed,
      'agreement':ref is not None and ref==proposed,
      'detected_keeper_samples':entry.get('keeper_samples_detected')})
mismatches=[x for x in results if not x['agreement']]
print('Metadata-to-goalkeeper agreements:',len(results)-len(mismatches),'/',len(results))
print('Mismatches:',mismatches[:10])
if mismatches:
    raise AssertionError('Attacking orientation disagrees with metadata: STOP and inspect before deriving progression')


Metadata-to-goalkeeper agreements: 80 / 80
Mismatches: []


## 2 — Compute source-oriented candidate endpoint differences

Each canonical row contains all valid teammate positions separated into directly detected and extrapolated. Write **one row per teammate candidate**; no trajectory, pass trajectory, interception probability or hypothetical carry is estimated. `signed_dx_m` is the X displacement toward the attacking goal. `dy_m` remains pitch-global Y displacement, while `pass_distance_m` is rotation invariant. For plotting consistently in one attacking orientation, `(attack_x, attack_y)=(sign*x, sign*y)` corresponds to a 180° rotation for teams attacking leftward, as in SkillCorner's tutorial.

The endpoint of a teammate's current position is not a verified receiver location at ball arrival.


In [3]:
totals=Counter()
per_match=[]
for mid in sorted({r['match_id'] for r in results}):
    path=OUT/f'P1_canonical_decisions_match_{mid}_ROLES.jsonl'
    if not path.is_file(): raise FileNotFoundError(path)
    n=Counter()
    dest=OUT/f'P2_signed_candidate_geometry_match_{mid}.jsonl'
    with path.open(encoding='utf-8') as f, dest.open('w',encoding='utf-8') as g:
        for line in f:
            row=json.loads(line)
            n['decisions']+=1
            c=row.get('carrier_xy')
            if c is None:
                n['carrier_xy_missing']+=1
                continue
            sign=metadata_map.get((mid,int(row['period']),str(row['attacking_team_id'])))
            if sign not in (-1,1): raise AssertionError(f'{mid}: missing direction for {row["attacking_team_id"]}')
            try:
                cx,cy=map(float,c)
                if not all(map(math.isfinite,(cx,cy))): raise ValueError()
            except (TypeError,ValueError):
                n['carrier_xy_invalid']+=1
                continue
            n['eligible_decisions']+=1
            for quality in ('detected','extrapolated'):
                for teammate in row.get(f'teammates_{quality}',[]):
                    xy=teammate.get('xy')
                    if xy is None:continue
                    tx,ty=map(float,xy)
                    if not math.isfinite(tx) or not math.isfinite(ty):continue
                    dx=sign*(tx-cx)
                    dy=ty-cy
                    record={'match_id':mid,'period':row['period'],'frame':row['frame'],
                        'possession_event_id':row['possession_event_id'],
                        'carrier_player_id':row['carrier_player_id'],
                        'teammate_player_id':teammate['player_id'],
                        'attacking_team_id':row['attacking_team_id'],
                        'source_attack_x_sign':sign,
                        'carrier_detected':row.get('carrier_detected'),
                        'teammate_quality':quality,
                        'signed_dx_m':dx,'lateral_dy_m':dy,
                        'pass_distance_m':math.hypot(tx-cx,ty-cy),
                        'source_orientation':'match.json home_team_side'}
                    g.write(json.dumps(record,separators=(',',':'))+'\n')
                    n['candidate_pairs']+=1
                    n['positive_dx']+=dx>0
                    n['zero_dx']+=dx==0
                    n['negative_dx']+=dx<0
    if n['eligible_decisions']+n['carrier_xy_missing']+n['carrier_xy_invalid']!=n['decisions']:
        raise AssertionError(f'{mid}: denominator consistency failure')
    totals.update(n)
    per_match.append({'match_id':mid,**dict(n)})
    print(mid,'decisions',n['decisions'],'candidates',n['candidate_pairs'])


1874553 decisions 959 candidates 9560
1886347 decisions 999 candidates 9990
1899585 decisions 929 candidates 9290
1925299 decisions 1062 candidates 10620
1927964 decisions 1140 candidates 11390
1953632 decisions 979 candidates 9700
1959846 decisions 946 candidates 9450
1986691 decisions 915 candidates 9150
1996435 decisions 1005 candidates 9970
1996436 decisions 786 candidates 7850
2006229 decisions 979 candidates 9770
2006363 decisions 976 candidates 9760
2007448 decisions 863 candidates 8620
2007721 decisions 959 candidates 9590
2010085 decisions 935 candidates 9350
2011166 decisions 757 candidates 7550
2013725 decisions 922 candidates 9170
2015213 decisions 883 candidates 8810
2016236 decisions 959 candidates 9590
2017461 decisions 839 candidates 8390


In [4]:
# Fail fast if any generated file is not genuine one-record-per-line JSONL.
for mid in sorted({r['match_id'] for r in results}):
    p=OUT/f'P2_signed_candidate_geometry_match_{mid}.jsonl'
    count=0
    with p.open(encoding='utf-8') as stream:
        for line_no,line in enumerate(stream,1):
            try: item=json.loads(line)
            except json.JSONDecodeError as err:
                raise AssertionError(f'Malformed JSONL in {p.name} at physical line {line_no}: {err}') from err
            if not isinstance(item,dict):raise AssertionError(f'Non-object JSONL row in {p.name}: {line_no}')
            count+=1
    expected=next(x['candidate_pairs'] for x in per_match if x['match_id']==mid)
    assert count==expected,(mid,count,expected)
print('All 20 signed-candidate files contain valid line-delimited JSON objects.')

assert totals['decisions']==18792,totals
assert totals['eligible_decisions']==18757,totals
assert totals['candidate_pairs']==187570,totals
assert len(per_match)==20
gate={'created_utc':datetime.now(timezone.utc).isoformat(),
  'purpose':'Official-metadata-verified direction and source-oriented static teammate geometry',
  'metadata_goalkeeper_comparisons':len(results),
  'metadata_goalkeeper_agreements':len(results)-len(mismatches),
  'metadata_goalkeeper_disagreements':mismatches,
  'totals':dict(totals),
  'per_match':per_match,
  'interpretation':[
    'home_team_side in match.json is authoritative direction; goalkeeper analysis is an independent cross-check',
    'positive signed_dx is endpoint projection toward attacking goal, not a feasible or observed pass',
    'No observed action outcomes, vendor end states or future frames are model predictors',
    'No arbitrary forward/backward threshold; signed displacement retained continuously',
    'Coordinates from players current at possession start; not receiver destination at arrival',
    'Carrier and target detections/extrapolations kept separate',
    'Per-match candidate files saved locally']}
target=OUT/'P2_signed_progression_gate.json'
target.write_text(json.dumps(gate,indent=2),encoding='utf-8')
print(json.dumps({'agreement':f'{len(results)-len(mismatches)}/{len(results)}','totals':dict(totals)},indent=2))
print('Upload:',target)


All 20 signed-candidate files contain valid line-delimited JSON objects.
{
  "agreement": "80/80",
  "totals": {
    "decisions": 18792,
    "eligible_decisions": 18757,
    "candidate_pairs": 187570,
    "positive_dx": 103565,
    "zero_dx": 44,
    "negative_dx": 83961,
    "carrier_xy_missing": 35
  }
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_signed_progression_gate.json
